In [1]:
import pandas as pd
import os
from pathlib import Path
import re
from datetime import datetime
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import rasterio
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchmetrics.classification import MulticlassF1Score
from glob import glob
import torch
import random
import torch.nn.functional as F
import numpy as np
import torch.nn as nn

C:\Users\lolke\miniconda3\envs\cosmohack\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
s1_pre_folder = './bs/sentinel1_pre/'
s1_post_folder = './bs/sentinel1_post/'
s2_pre_folder = './bs/sentinel2_pre/'
s2_post_folder = './bs/sentinel2_post/'
aux_folder = './bs/aux/'

ds_metadata = pd.read_csv("./bs/meta.csv")

In [3]:
import rasterio
import numpy as np

idx = 21
s1_pre_prefix = "_Sentinel-1_pre"
s1_post_prefix = "_Sentinel-1_post"
s2_pre_prefix = "_Sentinel-2_pre"
s2_post_prefix = "_Sentinel-2_post"
aux_prefix = "_AUX"
path = aux_folder + ds_metadata.iloc[idx]["chip_id"] + aux_prefix + ".tif"

with rasterio.open(path) as src:
    s1_pre = src.read().astype(np.float32)  # shape: (C, H, W)
    print("shape:", s1_pre.shape, "dtype:", s1_pre.dtype)
    print("min/max:", s1_pre.min(), s1_pre.max())

shape: (3, 512, 512) dtype: float32
min/max: 4.0 141.0


In [4]:
import numpy as np
import rasterio

path = "bs/aux/BS_tr_000003_AUX.tif"  # подставь реальный путь

with rasterio.open(path) as src:
    aux = src.read().astype(np.float32)
    print("shape:", aux.shape, "dtype в файле:", src.dtypes, "nodata:", src.nodata)

for c in range(aux.shape[0]):
    ch = aux[c]
    finite = ch[np.isfinite(ch)]
    n_unique = len(np.unique(finite))
    print(f"\n--- канал {c} ---")
    print(f"min={finite.min():.2f} max={finite.max():.2f} mean={finite.mean():.2f}")
    print(f"n_unique={n_unique}")
    if n_unique <= 20:
        vals, counts = np.unique(finite, return_counts=True)
        print("уникальные значения и их частоты:", dict(zip(vals.tolist(), counts.tolist())))
    else:
        print("перцентили [1,25,50,75,99]:", np.percentile(finite, [1, 25, 50, 75, 99]))

shape: (3, 512, 512) dtype в файле: ('int16', 'int16', 'int16') nodata: None

--- канал 0 ---
min=-13.00 max=4.00 mean=-5.17
n_unique=18
уникальные значения и их частоты: {-13.0: 27532, -12.0: 523, -11.0: 2766, -10.0: 6764, -9.0: 14710, -8.0: 11386, -7.0: 13435, -6.0: 21161, -5.0: 25662, -4.0: 28859, -3.0: 36360, -2.0: 39721, -1.0: 21101, 0.0: 9513, 1.0: 1501, 2.0: 836, 3.0: 312, 4.0: 2}

--- канал 1 ---
min=1.00 max=1.00 mean=1.00
n_unique=1
уникальные значения и их частоты: {1.0: 262144}

--- канал 2 ---
min=20.00 max=90.00 mean=34.33
n_unique=6
уникальные значения и их частоты: {20.0: 104, 30.0: 227218, 40.0: 58, 60.0: 31517, 80.0: 469, 90.0: 2778}


In [89]:
import random
import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset

S2_SCALE = 10000.0
S1_SCALE = 100.0

S1_CLIP_RANGES = {"VV": (-25.0, 0.0), "VH": (-32.5, 0.0)}

S2_BAND_IDX = {
    "B2": 0, "B3": 1, "B4": 2, "B5": 3, "B6": 4,
    "B7": 5, "B8A": 6, "B11": 7, "B12": 8, "SCL": 9,
}

S2_DN_CLIP = (0.0, 10000.0)

SCL_INVALID = {0, 1, 3, 6, 8, 9, 10}

WORLDCOVER_CODES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
CODE_TO_CLASS_IDX = {code: i for i, code in enumerate(WORLDCOVER_CODES)}
N_LANDCOVER_CLASSES = len(WORLDCOVER_CODES)

LANDCOVER_GROUPS = {
    "forest_shrub": {
        "codes": {10, 20, 95},
        "thresholds": (0.10, 0.27, 0.66),
    },
    "grass_moss": {
        "codes": {30, 100},
        "thresholds": (0.062, 0.204, 0.386),
    },
    "cropland": {
        "codes": {40},
        "thresholds": (0.07, 0.17, 0.38),
    },
    "wetland": {
        "codes": {90},
        "thresholds": (0.075, 0.331, 0.677),
    },
}

LANDCOVER_GROUP_NAMES = [
    "forest_shrub",
    "grass_moss",
    "cropland",
    "wetland",
    "other",
]

N_LANDCOVER_GROUPS = len(LANDCOVER_GROUP_NAMES)
FOLDERS = {
    "s1_pre": ("./bs/sentinel1_pre/", "_Sentinel-1_pre"),
    "s1_post": ("./bs/sentinel1_post/", "_Sentinel-1_post"),
    "s2_pre": ("./bs/sentinel2_pre/", "_Sentinel-2_pre"),
    "s2_post": ("./bs/sentinel2_post/", "_Sentinel-2_post"),
    "aux": ("./bs/aux/", "_AUX"),
    "mask": ("./bs/masks/", "_MASK"),  # уточни реальную папку/суффикс маски у себя
}

def make_landcover_dnbr_features(land_cover_raw, dnbr):
    h, w = land_cover_raw.shape

    group_onehot = np.zeros(
        (N_LANDCOVER_GROUPS, h, w),
        dtype=np.float32,
    )

    threshold_distance = np.zeros(
        (3, h, w),
        dtype=np.float32,
    )

    known_group_mask = np.zeros(
        (h, w),
        dtype=bool,
    )

    dnbr_2d = dnbr[0]

    for group_idx, group_name in enumerate(
        LANDCOVER_GROUP_NAMES[:-1]
    ):
        group_info = LANDCOVER_GROUPS[group_name]

        group_codes = list(group_info["codes"])
        group_mask = np.isin(
            land_cover_raw,
            group_codes,
        )

        thresholds = np.array(
            group_info["thresholds"],
            dtype=np.float32,
        )

        group_onehot[group_idx] = group_mask.astype(
            np.float32
        )

        for threshold_idx in range(3):
            threshold_distance[
                threshold_idx,
                group_mask,
            ] = (
                dnbr_2d[group_mask]
                - thresholds[threshold_idx]
            )

        known_group_mask |= group_mask

    other_mask = ~known_group_mask

    group_onehot[-1] = other_mask.astype(
        np.float32
    )

    threshold_distance = np.clip(
        threshold_distance,
        -1.0,
        1.0,
    )

    return group_onehot, threshold_distance

def landcover_group_to_onehot(land_cover_raw):
    h, w = land_cover_raw.shape

    group_onehot = np.zeros(
        (5, h, w),
        dtype=np.float32,
    )

    forest_shrub_mask = np.isin(
        land_cover_raw,
        [10, 20, 95],
    )

    grass_moss_mask = np.isin(
        land_cover_raw,
        [30, 100],
    )

    cropland_mask = np.isin(
        land_cover_raw,
        [40],
    )

    wetland_mask = np.isin(
        land_cover_raw,
        [90],
    )

    other_mask = ~(
        forest_shrub_mask
        | grass_moss_mask
        | cropland_mask
        | wetland_mask
    )

    group_onehot[0] = forest_shrub_mask.astype(
        np.float32
    )

    group_onehot[1] = grass_moss_mask.astype(
        np.float32
    )

    group_onehot[2] = cropland_mask.astype(
        np.float32
    )

    group_onehot[3] = wetland_mask.astype(
        np.float32
    )

    group_onehot[4] = other_mask.astype(
        np.float32
    )

    return group_onehot


def build_path(chip_id, kind):
    folder, suffix = FOLDERS[kind]
    return folder + chip_id + suffix + ".tif"


def load_tif(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

def load_s2(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    arr = np.clip(arr, S2_DN_CLIP[0], S2_DN_CLIP[1])
    return arr

def normalize_s1_db_scaled(arr, band_indices={"VV": 0, "VH": 1}, scale=S1_SCALE):
    h, w = arr.shape[1], arr.shape[2]
    out = np.zeros((2, h, w), dtype=np.float32)
    for out_i, band in enumerate(("VV", "VH")):
        src_i = band_indices[band]
        band_db = arr[src_i] / scale
        lo, hi = S1_CLIP_RANGES[band]
        band_db = np.clip(band_db, lo, hi)
        out[out_i] = (band_db - lo) / (hi - lo)
    return out


def process_s1_pair(s1_pre_raw, s1_post_raw):
    return normalize_s1_db_scaled(s1_pre_raw), normalize_s1_db_scaled(s1_post_raw)


def build_valid_mask_scl(scl_pre, scl_post):
    invalid_pre = np.isin(scl_pre, list(SCL_INVALID))
    invalid_post = np.isin(scl_post, list(SCL_INVALID))
    return (~(invalid_pre | invalid_post)).astype(np.float32)


def process_aux_3ch(aux_arr, elevation_range):
    """aux_arr: (3, H, W) -> [dem, slope, land_cover_class]
    Порядок каналов подтверждён через rasterio descriptions: (dem, slope, landcover).
    Никакой маски валидности в AUX нет — валидность определяется только через SCL.
    """
    dem, slope, land_cover = aux_arr
    dem_norm = np.clip(
        (dem - elevation_range[0]) / (elevation_range[1] - elevation_range[0]),
        0, 1
    ).astype(np.float32)
    slope_norm = np.clip(slope / 90.0, 0, 1).astype(np.float32)  # градусы 0-90 -> [0,1]
    return dem_norm[None, ...], slope_norm[None, ...], land_cover.astype(np.int64)

IGNORE_INDEX = 255

class CosmohackDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def remap_mask(self, mask):
        return mask.astype(np.int64)  # BS-маска уже 0-3 согласно постановке кейса

    def apply_augmentations(self, x, mask):
        if random.random() > 0.5:
            x = np.flip(x, axis=2)
            mask = np.flip(mask, axis=1)
        if random.random() > 0.5:
            x = np.flip(x, axis=1)
            mask = np.flip(mask, axis=0)
        k = random.randint(0, 3)
        if k > 0:
            x = np.rot90(x, k, axes=(1, 2))
            mask = np.rot90(mask, k, axes=(0, 1))
        return np.ascontiguousarray(x), np.ascontiguousarray(mask)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        chip_id = row["chip_id"]
    
        s2_pre = load_s2(build_path(chip_id, "s2_pre")) / S2_SCALE
        s2_post = load_s2(build_path(chip_id, "s2_post")) / S2_SCALE
    
        idx_red = S2_BAND_IDX["B4"]
        idx_nir = S2_BAND_IDX["B8A"]
        idx_swir1 = S2_BAND_IDX["B11"]
        idx_swir2 = S2_BAND_IDX["B12"]
        idx_scl = S2_BAND_IDX["SCL"]
    
        pre_red = s2_pre[idx_red][None, ...]
        pre_nir = s2_pre[idx_nir][None, ...]
        pre_swir1 = s2_pre[idx_swir1][None, ...]
        pre_swir2 = s2_pre[idx_swir2][None, ...]
    
        post_red = s2_post[idx_red][None, ...]
        post_nir = s2_post[idx_nir][None, ...]
        post_swir1 = s2_post[idx_swir1][None, ...]
        post_swir2 = s2_post[idx_swir2][None, ...]
    
        scl_pre = (s2_pre[idx_scl] * S2_SCALE).round()
        scl_post = (s2_post[idx_scl] * S2_SCALE).round()
    
        pre_nbr1 = np.clip((pre_nir - pre_swir1) / (pre_nir + pre_swir1 + 1e-8), -1.0, 1.0,)
    
        post_nbr1 = np.clip((post_nir - post_swir1) / (post_nir + post_swir1 + 1e-8), -1.0, 1.0,)
    
        pre_nbr2 = np.clip((pre_nir - pre_swir2) / (pre_nir + pre_swir2 + 1e-8), -1.0, 1.0,)
    
        post_nbr2 = np.clip((post_nir - post_swir2) / (post_nir + post_swir2 + 1e-8), -1.0, 1.0)
    
        pre_ndvi = np.clip((pre_nir - pre_red) / (pre_nir + pre_red + 1e-8),-1.0, 1.0,)
    
        post_ndvi = np.clip((post_nir - post_red) / (post_nir + post_red + 1e-8),-1.0, 1.0,)
    
        dndvi = np.clip(pre_ndvi - post_ndvi, -2.0, 2.0)
    
        dnbr1 = np.clip(pre_nbr1 - post_nbr1, -2.0, 2.0)
        dnbr2 = np.clip(pre_nbr2 - post_nbr2, -2.0, 2.0)
    
        rdnbr = dnbr1 / (np.sqrt(np.abs(pre_nbr1)) + 1e-8)
        rdnbr = np.clip(rdnbr, -6.0, 6.0) / 6.0
    
        scl_valid_mask = build_valid_mask_scl(scl_pre, scl_post)[None, ...]
    
        s1_pre_raw = load_tif(build_path(chip_id, "s1_pre"))
        s1_post_raw = load_tif(build_path(chip_id, "s1_post"))
    
        pre_norm_s1, post_norm_s1 = process_s1_pair(
            s1_pre_raw,
            s1_post_raw,
        )
    
        aux_raw = load_tif(build_path(chip_id, "aux"))

        land_cover_raw = aux_raw[2].astype(np.int64)
        
        landcover_group_onehot = landcover_group_to_onehot(land_cover_raw)
    
        combined_valid_mask = scl_valid_mask
        x = np.concatenate([
            pre_red,
            pre_nir,
            pre_swir1,
            pre_swir2,
            
            post_red,
            post_nir,
            post_swir1,
            post_swir2,
    
            # dndvi,
            # dnbr1,
            # dnbr2,
            rdnbr,
    
            # pre_norm_s1,
            # post_norm_s1,
    
            # dem_norm,
            # slope_norm,
    
            # land_cover_onehot,
            # combined_valid_mask,
            # threshold_distance,
            # landcover_group_onehot,
        ], axis=0).astype(np.float32)
        mask_path = build_path(chip_id, "mask")
        mask = load_tif(mask_path).astype(np.int64)[0]
        mask = self.remap_mask(mask)
    
        valid_pixels = combined_valid_mask[0] > 0.5
    
        mask[~valid_pixels] = IGNORE_INDEX
    
        optical_channels = slice(0, 10)
    
        x[optical_channels, ~valid_pixels] = 0.0
    
        x = np.nan_to_num(
            x,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(np.float32)
    
        if self.is_train:
            x, mask = self.apply_augmentations(x, mask)
    
        x_tensor = torch.from_numpy(x).float()
        mask_tensor = torch.from_numpy(mask).long()
    
        return x_tensor, mask_tensor

In [90]:
device = "cuda"

f1_metric = MulticlassF1Score(
    num_classes=4,
    average=None,
    ignore_index=255
).to(device)

In [91]:
class ConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.conv = nn.Sequential(
                                  nn.Conv2d(ch_in, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
                                  nn.Conv2d(ch_out, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = self.conv(x)
        return x

In [92]:
class UpConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.up = nn.Sequential(
                                nn.Upsample(scale_factor=2),
                                nn.Conv2d(ch_in, ch_out,
                                         kernel_size=3,stride=1,
                                         padding=1, bias=True),
                                nn.BatchNorm2d(ch_out),
                                nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = x = self.up(x)
        return x

In [93]:
class AttentionBlock(torch.nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        
        self.w_g = nn.Sequential(
                                nn.Conv2d(f_g, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.w_x = nn.Sequential(
                                nn.Conv2d(f_l, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.psi = nn.Sequential(
                                nn.Conv2d(f_int, 1,
                                         kernel_size=1, stride=1,
                                         padding=0,  bias=True),
                                nn.BatchNorm2d(1),
                                nn.Sigmoid(),
        )
        
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, g, x):
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = self.relu(g1+x1)
        psi = self.psi(psi)
        
        return psi*x

In [95]:
class AttentionUNet(torch.nn.Module):
    def __init__(self, n_classes=4, in_channel=9, out_channel=4):
        super().__init__() 
        
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv1 = ConvBlock(ch_in=in_channel, ch_out=64)
        self.conv2 = ConvBlock(ch_in=64, ch_out=128)
        self.conv3 = ConvBlock(ch_in=128, ch_out=256)
        self.conv4 = ConvBlock(ch_in=256, ch_out=512)
        self.conv5 = ConvBlock(ch_in=512, ch_out=1024)
        
        self.up5 = UpConvBlock(ch_in=1024, ch_out=512)
        self.att5 = AttentionBlock(f_g=512, f_l=512, f_int=256)
        self.upconv5 = ConvBlock(ch_in=1024, ch_out=512)
        
        self.up4 = UpConvBlock(ch_in=512, ch_out=256)
        self.att4 = AttentionBlock(f_g=256, f_l=256, f_int=128)
        self.upconv4 = ConvBlock(ch_in=512, ch_out=256)
        
        self.up3 = UpConvBlock(ch_in=256, ch_out=128)
        self.att3 = AttentionBlock(f_g=128, f_l=128, f_int=64)
        self.upconv3 = ConvBlock(ch_in=256, ch_out=128)
        
        self.up2 = UpConvBlock(ch_in=128, ch_out=64)
        self.att2 = AttentionBlock(f_g=64, f_l=64, f_int=32)
        self.upconv2 = ConvBlock(ch_in=128, ch_out=64)
        
        self.conv_1x1 = nn.Conv2d(64, out_channel,
                                  kernel_size=1, stride=1, padding=0)
        
    def forward(self, x):
        # encoder
        x1 = self.conv1(x)
        
        x2 = self.maxpool(x1)
        x2 = self.conv2(x2)
        
        x3 = self.maxpool(x2)
        x3 = self.conv3(x3)
        
        x4 = self.maxpool(x3)
        x4 = self.conv4(x4)
        
        x5 = self.maxpool(x4)
        x5 = self.conv5(x5)
        
        # decoder + concat
        d5 = self.up5(x5)
        x4 = self.att5(g=d5, x=x4)
        d5 = torch.concat((x4, d5), dim=1)
        d5 = self.upconv5(d5)
        
        d4 = self.up4(d5)
        x3 = self.att4(g=d4, x=x3)
        d4 = torch.concat((x3, d4), dim=1)
        d4 = self.upconv4(d4)
        
        d3 = self.up3(d4)
        x2 = self.att3(g=d3, x=x2)
        d3 = torch.concat((x2, d3), dim=1)
        d3 = self.upconv3(d3)
        
        d2 = self.up2(d3)
        x1 = self.att2(g=d2, x=x1)
        d2 = torch.concat((x1, d2), dim=1)
        d2 = self.upconv2(d2)
        
        d1 = self.conv_1x1(d2)
        
        return d1

In [96]:
from typing import Optional, Sequence
from torch import Tensor

class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6, ignore_index=255):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        valid_mask = (target != self.ignore_index)

        safe_target = target.clone()
        safe_target[~valid_mask] = 0  # временно подменяем, чтобы one_hot не упал

        probs = F.softmax(logits, dim=1)
        target_oh = F.one_hot(safe_target, self.num_classes).permute(0, 3, 1, 2).float()

        valid_mask = valid_mask.unsqueeze(1).float()  # [B, 1, H, W]
        probs = probs * valid_mask
        target_oh = target_oh * valid_mask

        dims = (0, 2, 3)
        intersection = torch.sum(probs * target_oh, dims)
        cardinality = torch.sum(probs + target_oh, dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()

class CombinedLoss(nn.Module):
    def __init__(self, class_weights, num_classes, ce_w=0.5, dice_w=0.5, ignore_index=255):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
        self.dice = DiceLoss(num_classes, ignore_index=ignore_index)
        self.ce_w, self.dice_w = ce_w, dice_w

    def forward(self, logits, target):
        return self.ce_w * self.ce(logits, target) + self.dice_w * self.dice(logits, target)

class FocalLoss(nn.Module):
    """ Focal Loss, as described in https://arxiv.org/abs/1708.02002.

    It is essentially an enhancement to cross entropy loss and is
    useful for classification tasks when there is a large class imbalance.
    x is expected to contain raw, unnormalized scores for each class.
    y is expected to contain class labels.

    Shape:
        - x: (batch_size, C) or (batch_size, C, d1, d2, ..., dK), K > 0.
        - y: (batch_size,) or (batch_size, d1, d2, ..., dK), K > 0.
    """

    def __init__(self,
                 alpha: Optional[Tensor] = None,
                 gamma: float = 0.,
                 reduction: str = 'mean',
                 ignore_index: int = -100):
        """Constructor.

        Args:
            alpha (Tensor, optional): Weights for each class. Defaults to None.
            gamma (float, optional): A constant, as described in the paper.
                Defaults to 0.
            reduction (str, optional): 'mean', 'sum' or 'none'.
                Defaults to 'mean'.
            ignore_index (int, optional): class label to ignore.
                Defaults to -100.
        """
        if reduction not in ('mean', 'sum', 'none'):
            raise ValueError(
                'Reduction must be one of: "mean", "sum", "none".')

        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction

        self.nll_loss = nn.NLLLoss(
            weight=alpha, reduction='none', ignore_index=ignore_index)

    def __repr__(self):
        arg_keys = ['alpha', 'gamma', 'ignore_index', 'reduction']
        arg_vals = [self.__dict__[k] for k in arg_keys]
        arg_strs = [f'{k}={v!r}' for k, v in zip(arg_keys, arg_vals)]
        arg_str = ', '.join(arg_strs)
        return f'{type(self).__name__}({arg_str})'

    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        if x.ndim > 2:
            # (N, C, d1, d2, ..., dK) --> (N * d1 * ... * dK, C)
            c = x.shape[1]
            x = x.permute(0, *range(2, x.ndim), 1).reshape(-1, c)
            # (N, d1, d2, ..., dK) --> (N * d1 * ... * dK,)
            y = y.view(-1)

        unignored_mask = y != self.ignore_index
        y = y[unignored_mask]
        if y.numel() == 0:
            return x.sum() * 0.0
        x = x[unignored_mask]

        # compute weighted cross entropy term: -alpha * log(pt)
        # (alpha is already part of self.nll_loss)
        log_p = F.log_softmax(x, dim=-1)
        ce = self.nll_loss(log_p, y)

        # get true class column from each row
        all_rows = torch.arange(len(x), device=x.device)
        log_pt = log_p[all_rows, y]

        # compute focal term: (1 - pt)^gamma
        pt = log_pt.exp()
        focal_term = (1 - pt)**self.gamma

        # the full loss: -alpha * ((1 - pt)^gamma) * log(pt)
        loss = focal_term * ce

        if self.reduction == 'mean':
            loss = loss.mean()
        elif self.reduction == 'sum':
            loss = loss.sum()

        return loss


def focal_loss(alpha: Optional[Sequence] = None,
               gamma: float = 2,
               reduction: str = 'mean',
               ignore_index: int = 255,
               device='cuda',
               dtype=torch.float32) -> FocalLoss:
    """Factory function for FocalLoss.

    Args:
        alpha (Sequence, optional): Weights for each class. Will be converted
            to a Tensor if not None. Defaults to None.
        gamma (float, optional): A constant, as described in the paper.
            Defaults to 0.
        reduction (str, optional): 'mean', 'sum' or 'none'.
            Defaults to 'mean'.
        ignore_index (int, optional): class label to ignore.
            Defaults to -100.
        device (str, optional): Device to move alpha to. Defaults to 'cpu'.
        dtype (torch.dtype, optional): dtype to cast alpha to.
            Defaults to torch.float32.

    Returns:
        A FocalLoss object
    """
    if alpha is not None:
        if not isinstance(alpha, Tensor):
            alpha = torch.tensor(alpha)
        alpha = alpha.to(device=device, dtype=dtype)

    fl = FocalLoss(
        alpha=alpha,
        gamma=gamma,
        reduction=reduction,
        ignore_index=ignore_index)
    return fl


class OHEM(nn.Module):
    def __init__(
        self,
        keep_ratio=0.4,
        class_weights=None,
        ignore_index=255,
        min_kept=1,
    ):
        super().__init__()

        if not 0.0 < keep_ratio <= 1.0:
            raise ValueError(
                "keep_ratio должен быть в диапазоне (0, 1]."
            )

        self.keep_ratio = keep_ratio
        self.class_weights = class_weights
        self.ignore_index = ignore_index
        self.min_kept = min_kept

    def forward(self, logits, target):
        pixel_loss = F.cross_entropy(
            logits,
            target,
            weight=self.class_weights,
            ignore_index=self.ignore_index,
            reduction="none",
        )

        valid_mask = target != self.ignore_index
        valid_loss = pixel_loss[valid_mask]

        if valid_loss.numel() == 0:
            return logits.sum() * 0.0

        n_valid = valid_loss.numel()

        n_keep = max(
            int(n_valid * self.keep_ratio),
            self.min_kept,
        )

        n_keep = min(
            n_keep,
            n_valid,
        )

        hard_losses, _ = torch.topk(
            valid_loss,
            k=n_keep,
            largest=True,
            sorted=False,
        )

        return hard_losses.mean()

In [101]:
from torch.utils.data import WeightedRandomSampler

def calculate_sample_weights(df):   
    global_class_counts = np.zeros(4, dtype=np.int64)
    chip_stats = []
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]
        mask_path = build_path(chip_id, "mask") 
        
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
            
        counts = np.bincount(mask.ravel(), minlength=4)
        global_class_counts += counts
        chip_stats.append(counts)
    
    class_weights = 1.0 / (global_class_counts + 1)
    class_weights = class_weights / np.sum(class_weights)
    
    sample_weights = []
    for counts in chip_stats:
        weight = np.sum(counts * class_weights)
        sample_weights.append(weight)
        
    return sample_weights

def calculate_class_weights(df):
    counts = np.zeros(4, dtype=np.int64)

    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]

        mask = load_tif(build_path(chip_id, "mask"))[0].astype(np.int64)

        s2_pre = load_s2(build_path(chip_id, "s2_pre")) / S2_SCALE
        s2_post = load_s2(build_path(chip_id, "s2_post")) / S2_SCALE

        scl_pre = (s2_pre[S2_BAND_IDX["SCL"]] * S2_SCALE).round()
        scl_post = (s2_post[S2_BAND_IDX["SCL"]] * S2_SCALE).round()

        valid = build_valid_mask_scl(scl_pre, scl_post).astype(bool)

        valid_mask_values = mask[valid]
        counts += np.bincount(valid_mask_values, minlength=4)

    weights = counts.sum() / (len(counts) * np.maximum(counts, 1))
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32, device=device)

In [103]:
import pandas as pd

SPLIT_PATH = "bs_temporal_split_train_2019_2023_val_2024.csv"

saved_split = pd.read_csv(SPLIT_PATH)

print("Saved split:")
print(saved_split["split"].value_counts())

meta_with_dates = ds_metadata.copy()

meta_with_dates["date_post_dt"] = pd.to_datetime(meta_with_dates["date_post"], errors="coerce",)

meta_with_dates["year_post"] = meta_with_dates["date_post_dt"].dt.year

train_ids = saved_split.loc[saved_split["split"] == "train_temporal", "chip_id",]

val_ids = saved_split.loc[saved_split["split"] == "val_2024", "chip_id",]

train_temporal_df = meta_with_dates[meta_with_dates["chip_id"].isin(train_ids)].copy().reset_index(drop=True)

val_2024_df = meta_with_dates[meta_with_dates["chip_id"].isin(val_ids)].copy().reset_index(drop=True)

NUM_EPOCHS = 100

Saved split:
split
train_temporal    142
val_2024           82
Name: count, dtype: int64


In [104]:
train_ds = CosmohackDataset(train_temporal_df, is_train=True,)

val_ds = CosmohackDataset(val_2024_df, is_train=False,)

class_weights = calculate_class_weights(train_temporal_df)

train_sample_weights = calculate_sample_weights(train_temporal_df)

sampler = WeightedRandomSampler(weights=train_sample_weights, num_samples=len(train_sample_weights), replacement=True,)

train_loader = DataLoader(train_ds, batch_size=4, sampler=sampler, shuffle=False, num_workers=0, pin_memory=True)

val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0, pin_memory=True,)

print("Train dataset:", len(train_ds))
print("Val dataset:", len(val_ds))
print("Class weights:", class_weights.detach().cpu().numpy())

100%|██████████| 142/142 [00:00<00:00, 245.80it/s]

Train dataset: 142
Val dataset: 82
Class weights: [0.06004139 1.0758559  1.0559716  1.8081311 ]


In [105]:
NUM_CLASSES = 4
IGNORE_INDEX = 255
device = 'cuda'

best_bs_score = -float("inf")
best_miou_target = -float("inf")

amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

model = AttentionUNet().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4, nesterov=True)
criterion = OHEM()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=NUM_EPOCHS)

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device, memory_format=torch.channels_last, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        if amp_dtype == torch.bfloat16:
            loss.backward()
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        train_loss += loss.detach().item()
            
    model.eval()
    f1_metric.reset()
    val_loss = 0.0
    # IoU классов 0, 1, 2, 3.
    total_intersection = torch.zeros(NUM_CLASSES,device=device,)
    total_union = torch.zeros(NUM_CLASSES,device=device,)
    
    # Метрики бинарной задачи:
    # burn = target > 0
    burn_intersection = torch.tensor(0.0, device=device,)
    burn_union = torch.tensor(0.0, device=device,)
    burn_pred_pixels = torch.tensor(0.0, device=device,)
    burn_target_pixels = torch.tensor(0.0, device=device,)

    valid_pixels_total = torch.tensor(0.0, device=device,)
    
    val_loss = 0.0
    total_intersection = torch.zeros(4, device=device)
    total_union = torch.zeros(4, device=device)
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                outputs = model(images)
                val_loss += criterion(outputs, masks).item()
            preds = outputs.argmax(dim=1)
            f1_metric.update(outputs, masks)
            valid = masks != 255
            
            for c in range(NUM_CLASSES):
                pred_c = (preds == c) & valid
                target_c = (masks == c) & valid
                total_intersection[c] += (pred_c & target_c).sum()
                total_union[c] += (pred_c | target_c).sum()
                
            pred_burn = (preds > 0) & valid
            target_burn = (masks > 0) & valid
    
            burn_intersection += (pred_burn & target_burn).sum()
    
            burn_union += (pred_burn | target_burn).sum()
    
            burn_pred_pixels += pred_burn.sum()
            burn_target_pixels += target_burn.sum()

            valid_pixels_total += valid.sum()

    # IoU для классов 0, 1, 2, 3.
    iou_per_class = torch.where(total_union > 0, total_intersection / total_union, torch.full_like(total_union, float("nan")),).cpu().numpy()
    
    # Mean IoU severity-классов: 1, 2, 3.
    miou_severity = np.nanmean(iou_per_class[1:])
    
    # mIoU всех классов, включая фон.
    miou_all = np.nanmean(iou_per_class)
    
    # Бинарный IoU: burn против background.
    iou_burn = (burn_intersection / (burn_union + 1e-7)).item()
    
    # Дополнительная метрика: F1 именно для burned pixels.
    f1_burn = (2.0 * burn_intersection / (burn_pred_pixels + burn_target_pixels + 1e-7)).item()
    
    pred_burn_fraction = (burn_pred_pixels / valid_pixels_total).item()
    target_burn_fraction = (burn_target_pixels / valid_pixels_total).item()
    
    mean_ious = torch.where(total_union > 0, total_intersection / total_union, torch.full_like(total_union, float("nan"))).cpu().numpy()
    miou_target = np.nanmean(mean_ious[1:])
    f1_epoch = f1_metric.compute().cpu().numpy()
    val_loss /= len(val_loader)
    
    # Эта величина лежит в диапазоне [0, 0.65],
    # потому что 0.35 + 0.30 = 0.65.
    bs_score = (0.35 * iou_burn + 0.30 * miou_severity)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    
    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} | "
        f"LR: {current_lr:.7f}\n"
        f"Train Loss: {train_loss / len(train_loader):.4f} | "
        f"Val Loss: {val_loss:.4f}\n"
        f"IoU per class [0,1,2,3]: "
        f"{np.round(iou_per_class, 4)}\n"
        f"F1 per class [0,1,2,3]:  "
        f"{np.round(f1_epoch, 4)}\n"
        f"IoU burn: {iou_burn:.4f} | "
        f"F1 burn: {f1_burn:.4f}\n"
        f"mIoU all: {miou_all:.4f} | "
        f"mIoU severity [1-3]: {miou_severity:.4f}\n"
        f"Burn fraction: "
        f"pred={pred_burn_fraction:.4f}, "
        f"true={target_burn_fraction:.4f}\n"
        f"BSScore: {bs_score:.4f}"
    )

    if bs_score > best_bs_score:
        best_bs_score = bs_score
        best_miou_target = miou_severity
    
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "bs_score": best_bs_score,
                "iou_burn": iou_burn,
                "f1_burn": f1_burn,
                "miou_severity": miou_severity,
                "iou_per_class": iou_per_class,
                "class_weights": class_weights.detach().cpu(),
            },
            "best_temporal_comboloss_sampler_rdnbr_OHEM.pt",
        )
    
        print(
            f"Checkpoint saved: "
            f"BSScore={best_bs_score:.4f} | "
            f"IoU_burn={iou_burn:.4f} | "
            f"mIoU_severity={miou_severity:.4f}"
        )

  0%|          | 0/100 [00:00<?, ?it/s]


Epoch 1/100 | LR: 0.0009998
Train Loss: 1.2736 | Val Loss: 1.0074
IoU per class [0,1,2,3]: [0.9102 0.     0.0044 0.024 ]
F1 per class [0,1,2,3]:  [0.953  0.     0.0088 0.0469]
IoU burn: 0.0474 | F1 burn: 0.0905
mIoU all: 0.2347 | mIoU severity [1-3]: 0.0095
Burn fraction: pred=0.0059, true=0.0924
BSScore: 0.0194


  1%|          | 1/100 [00:47<1:18:36, 47.64s/it]

Checkpoint saved: BSScore=0.0194 | IoU_burn=0.0474 | mIoU_severity=0.0095

Epoch 2/100 | LR: 0.0009990
Train Loss: 1.0547 | Val Loss: 0.7329
IoU per class [0,1,2,3]: [9.424e-01 3.000e-04 3.424e-01 5.845e-01]
F1 per class [0,1,2,3]:  [9.704e-01 7.000e-04 5.102e-01 7.378e-01]
IoU burn: 0.4932 | F1 burn: 0.6606
mIoU all: 0.4674 | mIoU severity [1-3]: 0.3091
Burn fraction: pred=0.0683, true=0.0924
BSScore: 0.2653


  2%|▏         | 2/100 [01:38<1:21:10, 49.70s/it]

...


Epoch 98/100 | LR: 0.0000010
Train Loss: 0.3387 | Val Loss: 0.4220
IoU per class [0,1,2,3]: [0.9471 0.2503 0.4308 0.6987]
F1 per class [0,1,2,3]:  [0.9729 0.4004 0.6021 0.8226]
IoU burn: 0.5421 | F1 burn: 0.7031
mIoU all: 0.5817 | mIoU severity [1-3]: 0.4599
Burn fraction: pred=0.0752, true=0.0924
BSScore: 0.3277


 99%|█████████▉| 99/100 [1:17:03<00:46, 46.10s/it]


Epoch 99/100 | LR: 0.0000002
Train Loss: 0.3686 | Val Loss: 0.4256
IoU per class [0,1,2,3]: [0.9479 0.2518 0.4335 0.7187]
F1 per class [0,1,2,3]:  [0.9732 0.4023 0.6048 0.8363]
IoU burn: 0.5539 | F1 burn: 0.7129
mIoU all: 0.5880 | mIoU severity [1-3]: 0.4680
Burn fraction: pred=0.0782, true=0.0924
BSScore: 0.3343


100%|██████████| 100/100 [1:17:49<00:00, 46.70s/it]


Epoch 100/100 | LR: 0.0000000
Train Loss: 0.3511 | Val Loss: 0.4224
IoU per class [0,1,2,3]: [0.9473 0.2344 0.4206 0.7035]
F1 per class [0,1,2,3]:  [0.9729 0.3798 0.5921 0.826 ]
IoU burn: 0.5347 | F1 burn: 0.6968
mIoU all: 0.5764 | mIoU severity [1-3]: 0.4528
Burn fraction: pred=0.0716, true=0.0924
BSScore: 0.3230


In [5]:
import numpy as np
import rasterio
import glob

def load_mask(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.int64)[0]

def class_distribution(mask_paths, n_classes=4):
    counts = np.zeros(n_classes, dtype=np.int64)
    for p in mask_paths:
        mask = load_mask(p)
        for c in range(n_classes):
            counts[c] += (mask == c).sum()
    return counts

mask_paths = glob.glob("./bs/masks/*.tif")
counts = class_distribution(mask_paths)
shares = counts / counts.sum()

for c in range(len(counts)):
    print(f"class {c}: {counts[c]} px ({shares[c]:.4%})")

class 0: 52811255 px (89.9370%)
class 1: 2362871 px (4.0239%)
class 2: 2192271 px (3.7334%)
class 3: 1353859 px (2.3056%)


In [22]:
def evaluate_model(model, loader, device, num_classes=4, ignore_index=255):
    model.eval()

    total_intersection = torch.zeros(num_classes, device=device)
    total_union = torch.zeros(num_classes, device=device)

    f1_metric = MulticlassF1Score(
        num_classes=num_classes,
        average=None,
        ignore_index=ignore_index,
    ).to(device)

    f1_metric.reset()

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(
                device,
                memory_format=torch.channels_last,
                non_blocking=True,
            )

            masks = masks.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype,
            ):
                outputs = model(images)

            preds = outputs.argmax(dim=1)

            f1_metric.update(outputs, masks)

            valid = masks != ignore_index

            for c in range(num_classes):
                pred_c = (preds == c) & valid
                target_c = (masks == c) & valid

                total_intersection[c] += (pred_c & target_c).sum()
                total_union[c] += (pred_c | target_c).sum()

    iou_per_class = torch.where(
        total_union > 0,
        total_intersection / total_union,
        torch.full_like(total_union, float("nan")),
    ).cpu().numpy()

    f1_per_class = f1_metric.compute().cpu().numpy()

    return {
        "iou_per_class": iou_per_class,
        "f1_per_class": f1_per_class,
        "miou_all": np.nanmean(iou_per_class),
        "miou_target": np.nanmean(iou_per_class[1:]),
    }
trained_model = AttentionUNet().to(device)
state_dict = torch.load("weighted_comboloss_sampler.pt")
trained_model.load_state_dict(state_dict)
trained_model.eval()
test_metrics = evaluate_model(
    model=trained_model,
    loader=test_loader,
    device=device,
    num_classes=4,
    ignore_index=IGNORE_INDEX,
)

print("Test F1 per class:", np.round(test_metrics["f1_per_class"], 4))
print("Test IoU per class:", np.round(test_metrics["iou_per_class"], 4))
print("Test mIoU all:", round(test_metrics["miou_all"], 4))
print("Test target mIoU:", round(test_metrics["miou_target"], 4))

Test F1 per class: [0.9368 0.3637 0.7188 0.7752]
Test IoU per class: [0.8811 0.2222 0.5611 0.633 ]
Test mIoU all: 0.5743
Test target mIoU: 0.4721


In [107]:
@torch.no_grad()
def get_confusion_matrix(
    model,
    loader,
    device,
    num_classes=4,
    ignore_index=255,
):
    model.eval()

    confusion = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.int64,
        device=device,
    )

    for images, masks in loader:
        images = images.to(
            device,
            memory_format=torch.channels_last,
            non_blocking=True,
        )

        masks = masks.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            logits = model(images)

        preds = logits.argmax(dim=1)

        valid = masks != ignore_index

        y_true = masks[valid].long()
        y_pred = preds[valid].long()

        indices = y_true * num_classes + y_pred

        confusion += torch.bincount(
            indices,
            minlength=num_classes * num_classes,
        ).reshape(num_classes, num_classes)

    return confusion.cpu().numpy()

In [108]:
cm = get_confusion_matrix(
    model=model,
    loader=val_loader,
    device=device,
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[12747405    70721   100699    34719]
 [  302004   127771    33074      183]
 [  148407    11284   252277    35389]
 [   52805        1    18715   336490]]


In [24]:
for col in ds_metadata.columns:
    col_lower = col.lower()

    if any(
        key in col_lower
        for key in [
            "date",
            "time",
            "year",
            "month",
            "lat",
            "lon",
            "lng",
            "x",
            "y",
            "region",
            "tile",
            "utm",
            "scene",
        ]
    ):
        print(f"\n{'=' * 70}")
        print(f"Column: {col}")
        print(f"dtype: {ds_metadata[col].dtype}")
        print(f"nunique: {ds_metadata[col].nunique(dropna=False)}")
        print(ds_metadata[col].head(10).tolist())


Column: region
dtype: float64
nunique: 1
[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]

Column: x_min
dtype: float64
nunique: 46
[491520.0, 471040.0, 655360.0, 614400.0, 481280.0, 686080.0, 337920.0, 358400.0, 583680.0, 440320.0]

Column: y_min
dtype: float64
nunique: 57
[5376000.0, 5068800.0, 5396480.0, 5345280.0, 5171200.0, 5099520.0, 5283840.0, 5324800.0, 5253120.0, 5570560.0]

Column: x_max
dtype: float64
nunique: 46
[501760.0, 481280.0, 665600.0, 624640.0, 491520.0, 696320.0, 348160.0, 368640.0, 593920.0, 450560.0]

Column: y_max
dtype: float64
nunique: 57
[5386240.0, 5079040.0, 5406720.0, 5355520.0, 5181440.0, 5109760.0, 5294080.0, 5335040.0, 5263360.0, 5580800.0]

Column: acq_datetime
dtype: float64
nunique: 1
[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]

Column: date_pre
dtype: str
nunique: 140
['2019-04-18', '2019-06-04', '2019-06-14', '2019-06-16', '2019-07-29', '2019-08-16', '2019-08-26', '2019-08-31', '2019-09-03', '2019-09-05']

Column: date_post
dtype: str
nun

In [25]:
meta_audit = ds_metadata.copy()

meta_audit["date_pre_dt"] = pd.to_datetime(
    meta_audit["date_pre"],
    errors="coerce",
)

meta_audit["date_post_dt"] = pd.to_datetime(
    meta_audit["date_post"],
    errors="coerce",
)

meta_audit["year_post"] = meta_audit["date_post_dt"].dt.year
meta_audit["month_post"] = meta_audit["date_post_dt"].dt.month

meta_audit["burn_px"] = (
    meta_audit["sev1_px"].fillna(0)
    + meta_audit["sev2_px"].fillna(0)
    + meta_audit["sev3_px"].fillna(0)
)

print("Диапазон date_pre:")
print(
    meta_audit["date_pre_dt"].min(),
    "—",
    meta_audit["date_pre_dt"].max(),
)

print("\nДиапазон date_post:")
print(
    meta_audit["date_post_dt"].min(),
    "—",
    meta_audit["date_post_dt"].max(),
)

print("\nСцены по post-fire году:")
display(
    meta_audit.groupby("year_post")
    .agg(
        n_chips=("chip_id", "count"),
        n_unique_x=("x_min", "nunique"),
        n_unique_y=("y_min", "nunique"),
        sev1_pixels=("sev1_px", "sum"),
        sev2_pixels=("sev2_px", "sum"),
        sev3_pixels=("sev3_px", "sum"),
        burn_pixels=("burn_px", "sum"),
    )
    .round(0)
)

print("\nСцены по post-fire году и месяцу:")
display(
    meta_audit.groupby(["year_post", "month_post"])
    .size()
    .reset_index(name="n_chips")
)

Диапазон date_pre:
2019-04-18 00:00:00 — 2024-10-11 00:00:00

Диапазон date_post:
2019-04-28 00:00:00 — 2024-11-02 00:00:00

Сцены по post-fire году:


,n_chips,n_unique_x,n_unique_y,sev1_pixels,sev2_pixels,sev3_pixels,burn_pixels
year_post,,,,,,,
2019,11,11,11,204583.0,164020.0,103448.0,472051.0
2020,71,32,41,788652.0,679671.0,431551.0,1899874.0
2021,11,11,11,175904.0,76155.0,31850.0,283909.0
2022,31,24,24,326753.0,507876.0,248762.0,1083391.0
2023,18,14,15,170624.0,156685.0,95686.0,422995.0
2024,82,33,41,696355.0,607864.0,442562.0,1746781.0



Сцены по post-fire году и месяцу:


,year_post,month_post,n_chips
0,2019,4,1
1,2019,6,3
2,2019,8,2
3,2019,9,4
4,2019,10,1
5,2020,4,4
6,2020,7,14
7,2020,8,16
8,2020,9,14
9,2020,10,21


In [53]:
from collections import defaultdict

def get_landcover_group(land_cover):
    group_idx = np.full(
        land_cover.shape,
        fill_value=4,
        dtype=np.int64,
    )

    group_idx[np.isin(land_cover, [10, 20, 95])] = 0
    group_idx[np.isin(land_cover, [30, 100])] = 1
    group_idx[np.isin(land_cover, [40])] = 2
    group_idx[np.isin(land_cover, [90])] = 3

    return group_idx

def inspect_dnbr_by_landcover(df, max_samples_per_chip=1000, seed=42):
    rng = np.random.default_rng(seed)

    group_names = [
        "forest_shrub",
        "grass_moss",
        "cropland",
        "wetland",
        "other",
    ]

    collected = defaultdict(list)

    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]

        s2_pre_raw = load_s2(
            build_path(chip_id, "s2_pre")
        )

        s2_post_raw = load_s2(
            build_path(chip_id, "s2_post")
        )

        aux_raw = load_tif(
            build_path(chip_id, "aux")
        )

        target = load_tif(
            build_path(chip_id, "mask")
        )[0].astype(np.int64)

        idx_nir = S2_BAND_IDX["B8A"]
        idx_swir2 = S2_BAND_IDX["B12"]
        idx_scl = S2_BAND_IDX["SCL"]

        pre_nir = s2_pre_raw[idx_nir] / S2_SCALE
        pre_swir2 = s2_pre_raw[idx_swir2] / S2_SCALE

        post_nir = s2_post_raw[idx_nir] / S2_SCALE
        post_swir2 = s2_post_raw[idx_swir2] / S2_SCALE

        pre_nbr = (
            ( pre_nir - pre_swir2 )
            / ( pre_nir + pre_swir2 + 1e-8 )
        )

        post_nbr = (
            ( post_nir - post_swir2 )
            / ( post_nir + post_swir2 + 1e-8 )
        )

        dnbr = np.clip(
            pre_nbr - post_nbr,
            -1.0,
            1.0,
        )

        scl_pre = s2_pre_raw[idx_scl].round()
        scl_post = s2_post_raw[idx_scl].round()

        valid_scl = build_valid_mask_scl(
            scl_pre,
            scl_post,
        ) > 0.5

        valid_target = (
            ( target >= 0 )
            & ( target < NUM_CLASSES )
        )

        valid = valid_scl & valid_target

        land_cover = aux_raw[2].astype(np.int64)
        group_idx = get_landcover_group(land_cover)

        for group_id, group_name in enumerate(group_names):
            for class_id in range(NUM_CLASSES):
                sample_mask = (
                    valid
                    & ( group_idx == group_id )
                    & ( target == class_id )
                )

                values = dnbr[sample_mask]

                if len(values) == 0:
                    continue

                if len(values) > max_samples_per_chip:
                    selected_idx = rng.choice(
                        len(values),
                        size=max_samples_per_chip,
                        replace=False,
                    )

                    values = values[selected_idx]

                collected[
                    (group_name, class_id)
                ].append(values.astype(np.float32))

    rows = []

    for group_name in group_names:
        for class_id in range(NUM_CLASSES):
            values_list = collected.get(
                (group_name, class_id),
                [],
            )

            if len(values_list) == 0:
                rows.append({
                    "landcover_group": group_name,
                    "target_class": class_id,
                    "n_sampled": 0,
                    "mean_dnbr": np.nan,
                    "p10": np.nan,
                    "p25": np.nan,
                    "p50": np.nan,
                    "p75": np.nan,
                    "p90": np.nan,
                })

                continue

            values = np.concatenate(values_list)

            rows.append({
                "landcover_group": group_name,
                "target_class": class_id,
                "n_sampled": len(values),
                "mean_dnbr": float(np.mean(values)),
                "p10": float(np.percentile(values, 10)),
                "p25": float(np.percentile(values, 25)),
                "p50": float(np.percentile(values, 50)),
                "p75": float(np.percentile(values, 75)),
                "p90": float(np.percentile(values, 90)),
            })

    result = pd.DataFrame(rows)

    return result

dnbr_stats = inspect_dnbr_by_landcover(
    train_temporal_df,
    max_samples_per_chip=1000,
    seed=42,
)

display(
    dnbr_stats.round(4)
)

100%|██████████| 142/142 [00:11<00:00, 11.97it/s]


,landcover_group,target_class,n_sampled,mean_dnbr,p10,p25,p50,p75,p90
0,forest_shrub,0,103502,0.0374,-0.0736,-0.0075,0.0382,0.0863,0.1429
1,forest_shrub,1,29635,0.1775,0.1045,0.1279,0.1681,0.2189,0.2659
2,forest_shrub,2,19248,0.3851,0.2547,0.3038,0.3736,0.4634,0.5394
3,forest_shrub,3,7827,0.6671,0.4953,0.5716,0.6608,0.7617,0.8592
4,grass_moss,0,137000,0.0080,-0.0848,-0.0273,0.0088,0.0459,0.0958
5,grass_moss,1,88714,0.1244,0.0677,0.0817,0.1122,0.1584,0.1939
6,grass_moss,2,72896,0.3022,0.2174,0.2504,0.3005,0.3499,0.3831
7,grass_moss,3,67905,0.5036,0.3939,0.4200,0.4712,0.5585,0.6672
8,cropland,0,122122,0.0171,-0.0979,-0.0325,0.0043,0.0533,0.1617
9,cropland,1,94657,0.1248,0.0756,0.0886,0.1154,0.1498,0.1737
